# DeepRacer-Genesis on Colab

AWS DeepRacer RL environment ported to [Genesis](https://github.com/Genesis-Embodied-AI/Genesis) — ROS-free, GPU-batched, trained with rsl-rl PPO.

This notebook installs the package, trains a driving policy with PPO on the GPU, and renders a bird's-eye video of all the agents driving at once.

**Runtime**: needs a GPU runtime (T4 works). Note that Colab VMs expose only the CUDA *compute* driver — no NVIDIA Vulkan/EGL graphics stack — so the Madrona/Nyx **camera-observation renderers cannot run here**; physics + training run on CUDA and the demo video renders through Mesa (software). For vision-based training use a machine with the full NVIDIA driver.

Each phase runs as a subprocess because Genesis initializes once per process.

In [ ]:
# @title GPU check
!nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv,noheader

Tesla T4, 580.82.07, 15360 MiB


In [ ]:
# @title Install deepracer-genesis
# Change this to your fork if needed. For local development you can instead
# upload a wheel and it will be picked up automatically.
REPO = "https://github.com/USERNAME/deepracer-genesis.git"  # <-- EDIT ME
BRANCH = "main"

import glob
local_wheel = sorted(glob.glob("/content/deepracer_genesis-*.whl"))
if local_wheel:
    print("installing local wheel:", local_wheel[-1])
    !pip install -q {local_wheel[-1]}
else:
    !pip install -q "deepracer-genesis @ git+{REPO}@{BRANCH}"
!pip show deepracer-genesis | head -2

installing local wheel: /content/deepracer_genesis-0.1.0-py3-none-any.whl


   ━━━━━━╸━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.3/88.0 MB 225.6 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━╺━━━━━━━━━━━━━━━ 53.6/88.0 MB 221.0 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 88.0/88.0 MB 176.5 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 88.0/88.0 MB 176.5 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 88.0/88.0 MB 176.5 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 88.0/88.0 MB 176.5 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 10.0 MB/s eta 0:00:00


Name: deepracer-genesis
Version: 0.1.0
ERROR: Pipe to stdout was broken
Exception ignored in: <_io.TextIOWrapper name='<stdout>' mode='w' encoding='utf-8'>
BrokenPipeError: [Errno 32] Broken pipe


In [ ]:
# @title Smoke test: the batched physics env builds and steps
%%writefile /content/smoke.py
import time
import torch
import genesis as gs
gs.init(backend=gs.cuda, logging_level="warning")
from deepracer_genesis.configs.cfgs import get_env_cfg
from deepracer_genesis.envs import DeepRacerEnv
env = DeepRacerEnv(num_envs=64, env_cfg=get_env_cfg(vision=False))
a = torch.zeros(64, 2, device=env.device)
for _ in range(20):
    env.step(a)
t0 = time.perf_counter()
for _ in range(200):
    env.step(a)
print("physics OK:", round(64 * 200 / (time.perf_counter() - t0)), "steps/s aggregate")

UsageError: Line magic function `%%writefile` not found.


UsageError: Line magic function `%%writefile` not found.

In [ ]:
# first run includes Genesis JIT compilation (a few minutes on a T4)
!python /content/smoke.py

python3: can't open file '/content/smoke.py': [Errno 2] No such file or directory


In [ ]:
# @title Train a driving policy (state-based PPO, with per-env domain randomization)
# ~10 min on a T4. Bump --max_iterations for confident lapping (300+).
!cd /content && python -m deepracer_genesis.train -B 512 --randomize --max_iterations 60 --exp_name colab 2>&1 | grep -E "Mean reward|episode length" | tail -6

                            Mean reward: 294.22
                    Mean episode length: 617.61
                            Mean reward: 299.90
                    Mean episode length: 623.86
                            Mean reward: 284.79
                    Mean episode length: 590.16


In [ ]:
# @title Render the many-agents demo video (bird's-eye, all cars at once)
# Colab renders this through Mesa (software): ~0.4 s/frame at 640x480 after a
# one-time ~35 s pipeline warmup, so 400 frames takes a few minutes.
import glob
ckpt = sorted(glob.glob("/content/logs/colab/model_*.pt"),
              key=lambda p: int(p.rsplit("_", 1)[1][:-3]))[-1]
print("checkpoint:", ckpt)
!cd /content && python -m deepracer_genesis.eval --checkpoint {ckpt} --num_envs 16 --steps 400 --res 640x480 --out logs/eval 2>&1 | tail -3

checkpoint: /content/logs/colab/model_59.pt


[Genesis] [21:12:09] [WARNING] This property is deprecated and will be removed in future release. Please use 'dofs_idx_local' instead.
[Genesis] [21:12:09] [WARNING] This property is deprecated and will be removed in future release. Please use 'dofs_idx_local' instead.
[Genesis] [21:12:09] [WARNING] This property is deprecated and will be removed in future release. Please use 'dofs_idx_local' instead.


In [ ]:
# @title Watch it
from IPython.display import Video
Video("/content/logs/eval/spectator.mp4", embed=True, width=720)

## Going further (on machines with a full NVIDIA driver)

Colab lacks the NVIDIA Vulkan/EGL graphics stack, so the camera-observation
renderers are unavailable here. On a workstation/server with the full driver:

- **Vision policy** (CNN on the 160x120 onboard camera, Madrona batch renderer):
  `python -m deepracer_genesis.train -B 128 --vision --max_iterations 300`
  (if the system CUDA toolkit is newer than 12.4, first run
  `scripts/fix_madrona_cuda13.sh` — aligns gs-madrona's bundled nvJitLink
  with a matching NVRTC)
- **Camera validation** (paired onboard + top-down images, 4 automated checks):
  `python -m deepracer_genesis.validation.camera_check --num_envs 4`
- **Heterogeneous tracks** (each env simulates a different track): pass
  `--tracks reinvent_base,reInvent2019_track,2022_reinvent_champ`
- **Domain randomization** follows the
  [Genesis DR guide](https://genesis-world.readthedocs.io/en/latest/user_guide/getting_started/policy_training/best_practices/domain_randomization.html):
  per-env friction/mass/COM per link, kp/kv/armature per dof (`--randomize`).
- **Throughput table**: `python benchmarks/throughput.py --sweep` (clone the repo).
- Branches: `raster-vision` (color-faithful rasterizer observations),
  `nyx-vision` (photorealistic Nyx renderer, driver 575+), `cpu-vision` (no GPU).